# Zero-shot vs Fine-tuned 베이스라인 비교

**목적**: 왜 fine-tuning이 필요한가를 데이터로 증명

| 조건 | 설명 |
|------|------|
| Zero-shot | KLUE-RoBERTa 기본 가중치로 추론 (파인튜닝 없음) |
| Fine-tuned | `klue_binary_final.pt` 재학습 모델로 추론 |

> **향후 개선사항**: Gemini 등 생성형 AI few-shot 비교 추가 예정

**평가 데이터**: `test_final.parquet`에서 정상/낚시성 균형 50건 랜덤 샘플링  
**지표**: Accuracy, F1-macro, Precision-macro, Recall-macro

In [1]:
!pip install -q transformers torch sentencepiece protobuf scikit-learn tqdm

## Step 1. Google Drive 마운트 + 경로 설정

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/text-mining-2026/data/processed'
SAVE_DIR = '/content/drive/MyDrive/text-mining-2026/models'

print(f'데이터 경로: {DATA_DIR}')
print(f'모델 경로:   {SAVE_DIR}')

Mounted at /content/drive
데이터 경로: /content/drive/MyDrive/text-mining-2026/data/processed
모델 경로:   /content/drive/MyDrive/text-mining-2026/models


## Step 2. 라이브러리 임포트

In [3]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 3. test_final에서 균형 50건 샘플링

정상(binary_label=0) 25건 + 낚시성(binary_label=1) 25건을 랜덤 추출합니다.  
세 조건(zero-shot / few-shot / fine-tuned) 모두 동일한 50건으로 평가합니다.

In [4]:
RANDOM_SEED = 42
N_SAMPLES   = 50   # 정상 25 + 낚시성 25

df_test = pd.read_parquet(f'{DATA_DIR}/test_final.parquet')
print(f'test_final 전체: {len(df_test):,}건')

df_normal    = df_test[df_test['binary_label'] == 0].sample(n=N_SAMPLES // 2, random_state=RANDOM_SEED)
df_clickbait = df_test[df_test['binary_label'] == 1].sample(n=N_SAMPLES // 2, random_state=RANDOM_SEED)
df_sample    = pd.concat([df_normal, df_clickbait]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'\n샘플 50건 구성:')
print(df_sample['binary_label'].value_counts().rename({0: '정상', 1: '낚시성'}).to_string())
print(f'\n컬럼: {list(df_sample.columns)}')
df_sample[['title_clean', 'content_clean', 'binary_label']].head(3)

test_final 전체: 36,434건

샘플 50건 구성:
binary_label
정상     25
낚시성    25

컬럼: ['newsID', 'newTitle', 'newsContent', 'binary_label', 'type_label', 'source_class', 'title_clean', 'content_clean']


,title_clean,content_clean,binary_label
0,유통가도 NFT 바람...자체 캐릭터로 MZ세대 공략,유통업계가 캐릭터를 활용한 대체불가토큰(NFT) 마케팅을 강화한다. 고객에게 색다른...,0
1,맥키스오페라 ‘뻔뻔한 클래식’ 섬 상륙,삼성전자가 Neo QLED 8K 등 올해 출시한 스마트 TV와 스마트 모니터에서 게...,1
2,'홍채'뚫린 갤럭시S8…현실에서 유출 가능성은?,KT가 외식업체 썬앳푸드와 손잡고 인공지능(AI) 기반의 외식업계 디지털 혁신(DX...,1


## Step 4. Dataset / 추론 함수 정의

In [5]:
MODEL_NAME = 'klue/roberta-base'
MAX_LENGTH = 512
BATCH_SIZE = 32

class ClickbaitDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.titles   = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels   = dataframe['binary_label'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            text=self.titles[idx],
            text_pair=self.contents[idx],
            truncation='only_second',
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


def run_inference(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = F.softmax(outputs.logits, dim=-1).cpu()
            preds   = torch.argmax(probs, dim=-1)
            all_preds.extend(preds.numpy())
            all_labels.extend(batch['labels'].numpy())
            all_probs.extend(probs[:, 1].numpy())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def compute_metrics(labels, preds, name=''):
    metrics = {
        'accuracy':        accuracy_score(labels, preds),
        'f1_macro':        f1_score(labels, preds, average='macro',     zero_division=0),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro':    recall_score(labels, preds, average='macro',  zero_division=0),
    }
    if name:
        print(f'\n[{name}]')
        for k, v in metrics.items():
            print(f'  {k:<22}: {v:.4f}')
    return metrics

print('Dataset / 추론 함수 정의 완료')

Dataset / 추론 함수 정의 완료


## Step 5. Zero-shot 추론

KLUE-RoBERTa 기본 가중치 그대로 사용 — 파인튜닝 없이 분류  
→ 모델이 낚시성 분류 학습을 전혀 하지 않은 상태의 성능

In [6]:
print('=== Zero-shot 추론 (파인튜닝 없음) ===')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = ClickbaitDataset(df_sample, tok)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# 파인튜닝된 가중치를 로드하지 않음 — HuggingFace 기본 가중치 그대로 사용
model_zs = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_zs.to(device)

labels_zs, preds_zs, probs_zs = run_inference(model_zs, loader)
metrics_zs = compute_metrics(labels_zs, preds_zs, name='Zero-shot (KLUE-RoBERTa 기본)')

del model_zs
torch.cuda.empty_cache()

=== Zero-shot 추론 (파인튜닝 없음) ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[Zero-shot (KLUE-RoBERTa 기본)]
  accuracy              : 0.5000
  f1_macro              : 0.3333
  precision_macro       : 0.2500
  recall_macro          : 0.5000


## Step 6. Fine-tuned 추론

`klue_binary_final.pt` — 전체 학습 데이터로 재학습한 최종 모델

In [7]:
print('=== Fine-tuned 추론 ===')

model_ft = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_ft.load_state_dict(torch.load(f'{SAVE_DIR}/pt_files/klue_binary_final.pt', map_location=device))
model_ft.to(device)

labels_ft, preds_ft, probs_ft = run_inference(model_ft, loader)
metrics_ft = compute_metrics(labels_ft, preds_ft, name='Fine-tuned (KLUE-RoBERTa)')

del model_ft
torch.cuda.empty_cache()

=== Fine-tuned 추론 ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[Fine-tuned (KLUE-RoBERTa)]
  accuracy              : 1.0000
  f1_macro              : 1.0000
  precision_macro       : 1.0000
  recall_macro          : 1.0000


## Step 9. 최종 비교표 및 성능 지표

## Step 7. 최종 비교표 및 성능 지표

In [8]:
METRIC_LABELS = {
    'accuracy':        'Accuracy',
    'f1_macro':        'F1-macro',
    'precision_macro': 'Precision-macro',
    'recall_macro':    'Recall-macro',
}

results = [
    ('Zero-shot\n(KLUE-RoBERTa)', metrics_zs),
    ('Fine-tuned\n(KLUE-RoBERTa)', metrics_ft),
]

rows = []
for model_name, m in results:
    row = {'모델': model_name}
    for k, label in METRIC_LABELS.items():
        row[label] = f"{m[k]:.4f}"
    rows.append(row)

df_result = pd.DataFrame(rows).set_index('모델')

print('=' * 60)
print('  Zero-shot vs Fine-tuned 비교 결과 (이진 분류, N=50)')
print('=' * 60)
print(df_result.to_string())
print('=' * 60)

  Zero-shot vs Fine-tuned 비교 결과 (이진 분류, N=50)
                           Accuracy F1-macro Precision-macro Recall-macro
모델                                                                       
Zero-shot\n(KLUE-RoBERTa)    0.5000   0.3333          0.2500       0.5000
Fine-tuned\n(KLUE-RoBERTa)   1.0000   1.0000          1.0000       1.0000


In [9]:
detail_rows = []
for i, (_, row) in enumerate(df_sample.iterrows()):
    detail_rows.append({
        '번호':        i + 1,
        '제목':        row['title_clean'][:40] + '...' if len(row['title_clean']) > 40 else row['title_clean'],
        '정답':        '낚시성' if row['binary_label'] == 1 else '정상',
        'Zero-shot':  '낚시성' if preds_zs[i] == 1 else '정상',
        'Fine-tuned': '낚시성' if preds_ft[i] == 1 else '정상',
    })

df_detail = pd.DataFrame(detail_rows)
df_detail['ZS 정오'] = df_detail.apply(lambda r: '✅' if r['Zero-shot']  == r['정답'] else '❌', axis=1)
df_detail['FT 정오'] = df_detail.apply(lambda r: '✅' if r['Fine-tuned'] == r['정답'] else '❌', axis=1)

print('정답률 요약:')
print(f"  Zero-shot  : {(df_detail['ZS 정오'] == '✅').mean() * 100:.1f}%")
print(f"  Fine-tuned : {(df_detail['FT 정오'] == '✅').mean() * 100:.1f}%")

display(df_detail)

정답률 요약:
  Zero-shot  : 50.0%
  Fine-tuned : 100.0%


,번호,제목,정답,Zero-shot,Fine-tuned,ZS 정오,FT 정오
0,1,유통가도 NFT 바람...자체 캐릭터로 MZ세대 공략,정상,정상,정상,✅,✅
1,2,맥키스오페라 ‘뻔뻔한 클래식’ 섬 상륙,낚시성,정상,낚시성,❌,✅
2,3,'홍채'뚫린 갤럭시S8…현실에서 유출 가능성은?,낚시성,정상,낚시성,❌,✅
3,4,행사 몰리는 10월... 대전 호텔업계 활짝 웃나?,낚시성,정상,낚시성,❌,✅
4,5,"李 \""마스크 잘 안 쓰죠?\"" 尹 \""작년부터 말 바꿔 믿기 힘들어\""",정상,정상,정상,✅,✅
5,6,[건강한 가족] 11가지 증상 개선하는 석류 농축액,낚시성,정상,낚시성,❌,✅
6,7,"'이마트, 겨울철 국민들 비타민 관리 위해 적자 감수하고 파격적 할인행사...",낚시성,정상,낚시성,❌,✅
7,8,"미국, 중국 굴기 막기 위해 인민군 약점 3가지 집중 공격해야",낚시성,정상,낚시성,❌,✅
8,9,"美국방부 \""레드플래그, 실제 상황과는 무관한 가상훈련\""",낚시성,정상,낚시성,❌,✅
9,10,법원서 대웅제약 리베이트 인정…의사 600명은 어쩌나?,정상,정상,정상,✅,✅


In [10]:
df_result.to_csv('zeroshot_vs_finetuned_metrics.csv', encoding='utf-8-sig')
df_detail.to_csv('zeroshot_vs_finetuned_detail.csv',  encoding='utf-8-sig', index=False)

print('저장 완료:')
print('  zeroshot_vs_finetuned_metrics.csv  — 성능 지표 요약')
print('  zeroshot_vs_finetuned_detail.csv   — 샘플별 예측 상세')

저장 완료:
  zeroshot_vs_finetuned_metrics.csv  — 성능 지표 요약
  zeroshot_vs_finetuned_detail.csv   — 샘플별 예측 상세


## Step 8. Classification Report

In [11]:
print('=== Zero-shot Classification Report ===')
print(classification_report(labels_zs, preds_zs,
                             target_names=['정상', '낚시성'], zero_division=0))

print('=== Fine-tuned Classification Report ===')
print(classification_report(labels_ft, preds_ft,
                             target_names=['정상', '낚시성'], zero_division=0))

=== Zero-shot Classification Report ===
              precision    recall  f1-score   support

          정상       0.50      1.00      0.67        25
         낚시성       0.00      0.00      0.00        25

    accuracy                           0.50        50
   macro avg       0.25      0.50      0.33        50
weighted avg       0.25      0.50      0.33        50

=== Fine-tuned Classification Report ===
              precision    recall  f1-score   support

          정상       1.00      1.00      1.00        25
         낚시성       1.00      1.00      1.00        25

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50



## 결과 요약

| 조건 | F1-macro | 해석 |
|------|----------|------|
| Zero-shot (KLUE-RoBERTa) | 0.33 | 파인튜닝 없이는 모든 기사를 낚시성으로 판정 — 사실상 분류 불가 |
| Fine-tuned (KLUE-RoBERTa) | ~1.00 | 학습 후 정상/낚시성 완벽 구분 (전체 test 기준 F1=0.9869) |

→ **fine-tuning의 필요성**: zero-shot 대비 F1 +0.66 이상 향상, 도메인 학습 없이는 BERT도 무용지물임을 수치로 증명

> **향후 개선사항**: Gemini / GPT 등 생성형 AI few-shot 비교 추가 시 더 풍부한 베이스라인 구성 가능